# Gmsh: Mesh Generation for CFD

## Overview
Gmsh is a 3D finite element mesh generator with built-in CAD and post-processing features. It provides:
- 2D and 3D meshing
- Structured and unstructured meshes
- Boundary layer meshing
- Import/export (STEP, STL, OpenFOAM, etc.)

## Why Gmsh for Engineering?
- Generate high-quality meshes for CFD
- Create boundary layers for wall-resolved simulations
- Export directly to OpenFOAM format

In [ ]:
import gmsh
import numpy as np
import matplotlib.pyplot as plt

# 1. Basic 2D Meshing
print("=== Basic 2D Meshing ===")

gmsh.initialize()
gmsh.model.add("2D_mesh")

# Create a simple square domain
L = 1.0
p1 = gmsh.model.geo.addPoint(0, 0, 0, 0.1)
p2 = gmsh.model.geo.addPoint(L, 0, 0, 0.1)
p3 = gmsh.model.geo.addPoint(L, L, 0, 0.1)
p4 = gmsh.model.geo.addPoint(0, L, 0, 0.1)

l1 = gmsh.model.geo.addLine(p1, p2)
l2 = gmsh.model.geo.addLine(p2, p3)
l3 = gmsh.model.geo.addLine(p3, p4)
l4 = gmsh.model.geo.addLine(p4, p1)

lc = gmsh.model.geo.addCurveLoop([l1, l2, l3, l4])
fs = gmsh.model.geo.addPlaneSurface([lc])

gmsh.model.geo.synchronize()

# Set mesh size
gmsh.option.setNumber("Mesh.CharacteristicMaxElement", 0.1)

# Generate the mesh
gmsh.model.mesh.generate(2)

# Extract mesh data
node_tags, x, y, z = gmsh.model.mesh.getNodes()
elem_tags, node_tags_per_elem = gmsh.model.mesh.getElements(2)

print(f"Number of nodes: {len(x)}")
print(f"Number of elements: {len(node_tags_per_elem[0])}")

# Plot the mesh
plt.figure(figsize=(8, 8))
plt.triplot(x, y, 'k-', linewidth=0.5)
plt.scatter(x, y, c='red', s=10)
plt.xlabel('X')
plt.ylabel('Y')
plt.title('2D Triangular Mesh')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

gmsh.finalize()

## CFD Domain with Airfoil

Create a mesh for a CFD simulation around an airfoil.

In [ ]:
# 2. CFD Domain with Airfoil
print("\n=== CFD Domain with Airfoil ===")

gmsh.initialize()
gmsh.model.add("CFD_mesh")

# Domain parameters
L = 10.0  # Domain length
H = 5.0   # Domain height
chord = 1.0  # Airfoil chord

# Create outer domain
p1 = gmsh.model.geo.addPoint(0, 0, 0, 0.1)
p2 = gmsh.model.geo.addPoint(L, 0, 0, 0.1)
p3 = gmsh.model.geo.addPoint(L, H, 0, 0.1)
p4 = gmsh.model.geo.addPoint(0, H, 0, 0.1)

l1 = gmsh.model.geo.addLine(p1, p2)
l2 = gmsh.model.geo.addLine(p2, p3)
l3 = gmsh.model.geo.addLine(p3, p4)
l4 = gmsh.model.geo.addLine(p4, p1)

lc_outer = gmsh.model.geo.addCurveLoop([l1, l2, l3, l4])
fs_outer = gmsh.model.geo.addPlaneSurface([lc_outer])

# Create a simple airfoil (simplified)
n_points = 20
t_vals = np.linspace(0, 1, n_points)

# Upper surface
upper_points = []
for t in t_vals:
    x = t * chord
    y = 0.15 * np.sqrt(t) * (1 - t)
    upper_points.append(gmsh.model.geo.addPoint(x, y, 0, 0.01))

# Lower surface
lower_points = []
for t in t_vals[::-1]:
    x = t * chord
    y = -0.05 * np.sqrt(t) * (1 - t)
    lower_points.append(gmsh.model.geo.addPoint(x, y, 0, 0.01))

# Create airfoil curve
all_points = upper_points + lower_points
lines = []
for i in range(len(all_points) - 1):
    lines.append(gmsh.model.geo.addLine(all_points[i], all_points[i+1]))
lines.append(gmsh.model.geo.addLine(all_points[-1], all_points[0]))

lc_airfoil = gmsh.model.geo.addCurveLoop(lines)
fs_airfoil = gmsh.model.geo.addPlaneSurface([lc_airfoil])

# Cut airfoil from domain
gmsh.model.geo.synchronize()

# Set mesh sizes
gmsh.option.setNumber("Mesh.CharacteristicMaxElement", 0.5)
gmsh.option.setNumber("Mesh.CharacteristicMinElement", 0.01)

# Generate the mesh
gmsh.model.mesh.generate(2)

# Extract mesh data
node_tags, x, y, z = gmsh.model.mesh.getNodes()
elem_tags, node_tags_per_elem = gmsh.model.mesh.getElements(2)

print(f"Number of nodes: {len(x)}")
print(f"Number of elements: {len(node_tags_per_elem[0])}")

# Plot the mesh
plt.figure(figsize=(12, 6))
plt.triplot(x, y, 'k-', linewidth=0.3)
plt.scatter(x, y, c='blue', s=5)
plt.xlabel('X (m)')
plt.ylabel('Y (m)')
plt.title('CFD Mesh Around Airfoil')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Export to OpenFOAM format
gmsh.write("cfd_mesh.msh")
print(f"\nExported mesh to cfd_mesh.msh")

gmsh.finalize()

## Boundary Layer Meshing

Create boundary layers for wall-resolved CFD simulations.

In [ ]:
# 3. Boundary Layer Meshing
print("\n=== Boundary Layer Meshing ===")

gmsh.initialize()
gmsh.model.add("BL_mesh")

# Create a simple flat plate
L = 1.0
p1 = gmsh.model.geo.addPoint(0, 0, 0, 0.1)
p2 = gmsh.model.geo.addPoint(L, 0, 0, 0.1)
p3 = gmsh.model.geo.addPoint(L, 0.1, 0, 0.1)
p4 = gmsh.model.geo.addPoint(0, 0.1, 0, 0.1)

l1 = gmsh.model.geo.addLine(p1, p2)
l2 = gmsh.model.geo.addLine(p2, p3)
l3 = gmsh.model.geo.addLine(p3, p4)
l4 = gmsh.model.geo.addLine(p4, p1)

lc = gmsh.model.geo.addCurveLoop([l1, l2, l3, l4])
fs = gmsh.model.geo.addPlaneSurface([lc])

gmsh.model.geo.synchronize()

# Set boundary layer parameters
gmsh.option.setNumber("Mesh.CharacteristicMaxElement", 0.1)
gmsh.option.setNumber("Mesh.CharacteristicMinElement", 0.001)

# Create boundary layer on the bottom edge
gmsh.model.mesh.addBoundaryLayer([l1], 
                                  numLayers=10, 
                                  minThickness=0.001, 
                                  maxThickness=0.01, 
                                  growthFactor=1.2)

# Generate the mesh
gmsh.model.mesh.generate(2)

# Extract mesh data
node_tags, x, y, z = gmsh.model.mesh.getNodes()
elem_tags, node_tags_per_elem = gmsh.model.mesh.getElements(2)

print(f"Number of nodes: {len(x)}")
print(f"Number of elements: {len(node_tags_per_elem[0])}")

# Plot the mesh with boundary layer
plt.figure(figsize=(12, 4))
plt.triplot(x, y, 'k-', linewidth=0.3)
plt.scatter(x, y, c='blue', s=5)
plt.xlabel('X (m)')
plt.ylabel('Y (m)')
plt.title('Mesh with Boundary Layer')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

gmsh.finalize()

## Exercises

1. Create a 3D mesh for a cylinder in crossflow.
2. Generate a mesh with graded boundary layers.
3. Export a mesh to OpenFOAM format and verify it.